# Inbound-callback shape-guard checks — M2-25, M2-26, M2-27, M2-28, M3-23, M3-24

Built 2026-08-15 (Batch 6). All six cases share one theme: an inbound ABDM callback (discover, link confirm, consent notify) shaped slightly differently than expected -- a missing/null field, a wrong type where a dict or list was expected, or a corrupted stored session/consent missing a field. For the two HIP-role callbacks that owe ABDM a synchronous-style acknowledgment (discover, consent notify), the real risk isn't a server crash -- Python exceptions inside these `async def` handlers are already caught by each function's own outer `try/except` -- it's a SILENTLY DROPPED acknowledgment: the exception fires after `mark_processed()`/before the real work but BEFORE `send_on_*()` is ever reached, so ABDM gets no response at all, just an eventual timeout, with only a log line as evidence. That's the same failure shape already found and fixed for M2-16 in Batch 5.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness
from unittest.mock import patch
import asyncio


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M2-25 / M2-26 — a discover callback with a missing/malformed `patient` field or wrongly-shaped identifier lists

**Real-world scenario**: ABDM's "discover patient" POST is missing the `patient` field entirely, or carries it as an explicit `null` (M2-25); or `verifiedIdentifiers`/`unverifiedIdentifiers` come back in an unexpected shape, or contain non-object entries (M2-26).

**Fix** (`server/callbacks/services/discover_service.py`): the OLD code took `body["patient"]` on faith (direct key access) and assumed every identifier entry was a dict -- a KeyError/AttributeError WAS caught by the outer try/except, but only after `save_patient_identity()`'s own crash point and BEFORE `send_on_discover()` (the ack) was ever reached. Now: `patient` defaults to `{}` for anything that isn't a dict (missing, null, or wrongly-shaped, logged clearly); `verifiedIdentifiers`/`unverifiedIdentifiers` default to `[]` if not a list; individual non-dict entries in either list are skipped rather than crashing on `.get()`. Processing continues far enough to still send ABDM a proper "no match" acknowledgment.

In [2]:
harness.activate_scratch_storage("m2_25_26")
import server.callbacks.services.discover_service as discover_service

for bad_body, label in [
    ({"headers": {"x-hip-id": "HIP-1", "request-id": "req-1"}, "body": {"transactionId": "txn-1"}}, "missing patient key (M2-25)"),
    ({"headers": {"x-hip-id": "HIP-1", "request-id": "req-2"}, "body": {"transactionId": "txn-2", "patient": None}}, "patient explicitly null (M2-25)"),
    ({"headers": {"x-hip-id": "HIP-1", "request-id": "req-3"}, "body": {"transactionId": "txn-3", "patient": {"verifiedIdentifiers": "not-a-list"}}}, "verifiedIdentifiers wrong shape (M2-26)"),
    ({"headers": {"x-hip-id": "HIP-1", "request-id": "req-4"}, "body": {"transactionId": "txn-4", "patient": {"verifiedIdentifiers": ["not-a-dict", {"type": "MOBILE", "value": "9999999999"}]}}}, "identifier list has non-dict entries (M2-26)"),
]:
    ack_recorder = harness.CallRecorder(harness.FakeResponse(202))
    with patch.object(discover_service, "search_patient", lambda **kw: []), \
         patch.object(discover_service, "send_on_discover", ack_recorder), \
         patch.object(discover_service, "save_patient_identity", lambda *a, **kw: None):
        try:
            await discover_service.process_discover(bad_body)
            crashed = False
        except Exception as exc:
            crashed = True
    harness.check(f"{label}: processed without crashing", not crashed)
    harness.check(f"{label}: ABDM still got an acknowledgment (not silently dropped)", ack_recorder.call_count == 1)


2026-08-15 19:18:29  -> Patient search request received from ABDM (POST /api/v3/hip/patient/care-context/discover)
2026-08-15 19:18:29  -> Extracted patient identifiers (ABHA address, mobile, MR if provided)
2026-08-15 19:18:29  -> No matching records found
2026-08-15 19:18:29     [API] Reporting Patient Match to ABDM -- POST .../on-discover -> 202


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_25_26_ez_lqu_g
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)


2026-08-15 19:18:29     [WAITING] Waiting for the patient to choose to link these records in the PHR app
2026-08-15 19:18:29  -> Patient search request received from ABDM (POST /api/v3/hip/patient/care-context/discover)
2026-08-15 19:18:29  -> Extracted patient identifiers (ABHA address, mobile, MR if provided)
2026-08-15 19:18:29  -> No matching records found
2026-08-15 19:18:29     [API] Reporting Patient Match to ABDM -- POST .../on-discover -> 202
2026-08-15 19:18:29     [WAITING] Waiting for the patient to choose to link these records in the PHR app
2026-08-15 19:18:29  -> Patient search request received from ABDM (POST /api/v3/hip/patient/care-context/discover)
2026-08-15 19:18:29     [ERROR] Discover request's 'verifiedIdentifiers' was not a list (got str) -- treating as empty rather than crashing.
2026-08-15 19:18:29  -> Extracted patient identifiers (ABHA address, mobile, MR if provided)
2026-08-15 19:18:29  -> No matching records found
2026-08-15 19:18:29     [API] Reporting 

PASS -- missing patient key (M2-25): processed without crashing
PASS -- missing patient key (M2-25): ABDM still got an acknowledgment (not silently dropped)
PASS -- patient explicitly null (M2-25): processed without crashing
PASS -- patient explicitly null (M2-25): ABDM still got an acknowledgment (not silently dropped)
PASS -- verifiedIdentifiers wrong shape (M2-26): processed without crashing
PASS -- verifiedIdentifiers wrong shape (M2-26): ABDM still got an acknowledgment (not silently dropped)
PASS -- identifier list has non-dict entries (M2-26): processed without crashing
PASS -- identifier list has non-dict entries (M2-26): ABDM still got an acknowledgment (not silently dropped)


---
## M3-23 / M3-24 — a consent notification with malformed `consentDetail` sub-fields

**Real-world scenario**: a GRANTED consent notification's `consentDetail.permission` is explicitly `null` (M3-24's own exact scenario), or `consentDetail.patient`/`consentDetail.hip`/`consentDetail.careContexts` are shaped unexpectedly (M3-23, generically).

**Fix** (`server/callbacks/services/consent_notify_service.py`): the OLD code chained `consent_detail.get("permission", {}).get("dateRange", {})`-style calls -- the `{}` default only covers a MISSING key, not one that's present but `null` or wrongly typed, so an explicit `null` or a wrong shape raised an uncaught `AttributeError` -- caught by the outer try/except, but only AFTER `mark_processed()` and BEFORE `send_on_consent_notify()` (the ack), same silently-dropped-ack shape as M2-16/M2-25/M2-26. A small local `_safe_dict()` helper now normalizes `patient`/`permission`/`hip` to a dict regardless of whether the real value was missing, null, or wrongly shaped (logged clearly), and `careContexts`/`hiTypes` are validated as lists the same way. Processing continues far enough to still send the ack.

In [3]:
harness.activate_scratch_storage("m3_23_24")
import server.callbacks.services.consent_notify_service as consent_notify_service

for bad_detail, label in [
    ({"permission": None}, "permission explicitly null (M3-24)"),
    ({"patient": "not-an-object"}, "patient wrong shape (M3-23)"),
    ({"hip": ["not", "an", "object"]}, "hip wrong shape (M3-23)"),
    ({"careContexts": "not-a-list"}, "careContexts wrong shape (M3-23)"),
]:
    callback_data = {
        "headers": {"request-id": f"req-{label}"},
        "body": {"notification": {"consentId": f"consent-{label}", "status": "GRANTED", "consentDetail": bad_detail}},
    }
    ack_recorder = harness.CallRecorder(harness.FakeResponse(202))
    with patch.object(consent_notify_service, "send_on_consent_notify", ack_recorder):
        try:
            await consent_notify_service.process_consent_notify(callback_data)
            crashed = False
        except Exception as exc:
            crashed = True
    harness.check(f"{label}: processed without crashing", not crashed)
    harness.check(f"{label}: ABDM still got an acknowledgment (not silently dropped)", ack_recorder.call_count == 1)


2026-08-15 19:19:32  -> Consent status update received from ABDM (POST /api/v3/consent/request/hip/notify)
2026-08-15 19:19:32  -> Extracted consent ID and status (GRANTED)
2026-08-15 19:19:32  -> Consent granted -- storing approved care contexts
2026-08-15 19:19:32     [API] Acknowledging Consent Notification to ABDM -- POST .../hip/on-notify -> 202
2026-08-15 19:19:32     [WAITING] Waiting for the HIU to request the patient's data
2026-08-15 19:19:32  -> Consent status update received from ABDM (POST /api/v3/consent/request/hip/notify)
2026-08-15 19:19:32  -> Extracted consent ID and status (GRANTED)
2026-08-15 19:19:32     [ERROR] Consent notification's 'consentDetail.patient' was not an object (got str) -- treating as empty rather than crashing.
2026-08-15 19:19:32  -> Consent granted -- storing approved care contexts
2026-08-15 19:19:32     [API] Acknowledging Consent Notification to ABDM -- POST .../hip/on-notify -> 202
2026-08-15 19:19:32     [WAITING] Waiting for the HIU to req

[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_23_24_m15mewfr
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- permission explicitly null (M3-24): processed without crashing
PASS -- permission explicitly null (M3-24): ABDM still got an acknowledgment (not silently dropped)
PASS -- patient wrong shape (M3-23): processed without crashing
PASS -- patient wrong shape (M3-23): ABDM still got an acknowledgment (not silently dropped)
PASS -- hip wrong shape (M3-23): processed without crashing
PASS -- hip wrong shape (M3-23): ABDM still got an acknowledgment (not silently dropped)
PASS -- careContexts wrong shape (M3-23): processed without crashing
PASS -- careContexts wrong shape (M3-23): ABDM still got an acknowledgment (not silently dropped)


---
## M2-27 — a link-confirmation message missing required fields (confirm-only, one bonus guard)

**Real-world scenario**: the `confirmation` object in a Link Confirm callback is missing entirely, present but empty, or shaped unexpectedly.

**Finding**: `link_confirm_service.py` already handled individually-missing fields cleanly -- `token`/`linkRefNumber` are both read via `.get()`, and a missing/unrecognized `linkRefNumber` already flows into a clean `"Link session not found -- cannot confirm."` log message, no crash either way. The only gap was `confirmation` itself being present but the WRONG type (a string, a list) instead of missing fields inside a dict -- `.get()` on a non-dict raises `AttributeError`. Added one small `isinstance` guard for that specific case, extending the same clean-error behavior consistently. No other fix needed -- this case's own described scenario (fields deliberately removed) was already handled correctly before this pass.

In [4]:
harness.activate_scratch_storage("m2_27")
import server.callbacks.services.link_confirm_service as link_confirm_service

for bad_body, label in [
    ({"headers": {"request-id": "req-m2-27-a"}, "body": {}}, "missing 'confirmation' key entirely"),
    ({"headers": {"request-id": "req-m2-27-b"}, "body": {"confirmation": {}}}, "confirmation present but empty"),
    ({"headers": {"request-id": "req-m2-27-c"}, "body": {"confirmation": "not-an-object"}}, "confirmation wrong shape"),
]:
    try:
        await link_confirm_service.process_link_confirm(bad_body)
        crashed = False
    except Exception as exc:
        crashed = True
    harness.check(f"{label}: processed without crashing (clean error, not a crash)", not crashed)


2026-08-15 19:20:11  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 19:20:11     [ERROR] Link session not found -- cannot confirm.
2026-08-15 19:20:11  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 19:20:11     [ERROR] Link session not found -- cannot confirm.
2026-08-15 19:20:11  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 19:20:11     [ERROR] Link confirm request's 'confirmation' field was not an object (got str) -- cannot confirm.


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_27_93z8wslv
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- missing 'confirmation' key entirely: processed without crashing (clean error, not a crash)
PASS -- confirmation present but empty: processed without crashing (clean error, not a crash)
PASS -- confirmation wrong shape: processed without crashing (clean error, not a crash)


---
## M2-28 — a corrupted saved link session missing a required field

**Real-world scenario**: a link session on disk is hand-edited/corrupted so it's missing a field (`otp_txn_id`, `abha_address`, or `selected_patient_records`) that Link Confirm needs, then the matching confirm callback arrives.

**Finding**: the OLD code read these via direct dict indexing (`session["otp_txn_id"]`, etc.) further down the function -- a missing field raised a `KeyError` that WAS already caught by the outer try/except (not a silent crash), with a message that happened to be reasonably clear (Python's own `KeyError` repr shows the missing key name) but wasn't this codebase's usual explicit, readable `log_error()` style.

**Fix**: added an explicit up-front check for all three required session fields, reporting exactly which field(s) are missing in one clear message instead of relying on an incidental exception message. Also added the same explicit check for `patient["hip_id"]` a few lines later (a corrupted stored patient identity record would have hit the identical problem one step later).

In [5]:
from server.callbacks.repository.link_repository import save_link_session

save_link_session("linkref-m2-28", {"abha_address": "test@sbx"})  # missing otp_txn_id, selected_patient_records

callback_data = {
    "headers": {"request-id": "req-m2-28"},
    "body": {"confirmation": {"token": "123456", "linkRefNumber": "linkref-m2-28"}},
}
try:
    await link_confirm_service.process_link_confirm(callback_data)
    crashed = False
except Exception as exc:
    crashed = True

harness.check("corrupted session (missing otp_txn_id/selected_patient_records) -> processed without crashing (clean, specific error)", not crashed)


2026-08-15 19:20:38  -> Patient submitted OTP -- confirming link (POST /api/v3/hip/link/care-context/confirm)
2026-08-15 19:20:38     [ERROR] Link session for linkref-m2-28 is missing required field(s) ['otp_txn_id', 'selected_patient_records'] -- cannot confirm (corrupted or incomplete session).


PASS -- corrupted session (missing otp_txn_id/selected_patient_records) -> processed without crashing (clean, specific error)


True